# Step 53 — Audit the exported SEISAN REA database

All diagnostic code is in this notebook. It reads S-files recursively from the configured REA directory and replaces daily CSV/PNG diagnostics in the external diagnostics directory. It does not modify S-files or rerun the catalogue/export.

## Classification and limitations

EVENT_CLASS comment records from step 52 are the preferred composition source, including P_H. If that comment is missing, the code uses legacy heuristics based on Nordic record contents and the configured waveform-name regex. Heuristics can mistake time-only/derived origin headers for real hypocenters, so inspect fallback-classified records rather than treating them as validated content. The canonical step-50 manifest remains the baseline for comparison.

W/P/H presence counts overlap: a W_P_H file contributes to all three presence totals but only one class. Do not stack those presence totals as disjoint categories or add them together. The daily date comes from the S-file filename, not its preferred origin; collision allocation in step 52 may move a filename to another day while preserving the scientific timestamps. These plots diagnose database export, not physical earthquake rates or clock accuracy.

Read errors are skipped by the existing implementation; current printed counts therefore do not constitute a complete unreadable-file audit. The plot retains the current diagnostic style. Review any class/count mismatch against step 52's failures and manifest before interpreting missing records scientifically.

## Configuration

Inspect all input/output paths and settings. This cell resolves configuration; definitions and run cells follow.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    rea_dir=P['FAIR_REA_DIR'],
    out_dir=P['SEISAN_DIAG_DIR'],
    wavefile_regex='\\b[MS]\\.[A-Za-z0-9]{1,12}_[A-Za-z0-9]{1,6}\\b',
)
display(SETTINGS)

## Workflow implementation

The next cell defines functions only. It does not execute the workflow or write outputs.

In [ ]:
from __future__ import annotations

import re

from pathlib import Path

from collections import Counter, defaultdict

import pandas as pd

import matplotlib.pyplot as plt

def iter_sfiles(rea_dir: Path):
    for p in rea_dir.rglob("*"):
        if p.is_file() and re.search(r"\.S\d{6}$", p.name):
            yield p

def get_event_class(lines):
    """
    Extract EVENT_CLASS from Nordic comment lines.
    Expected:
      EVENT_CLASS:W+P+H
    """
    for ln in lines:
        if "EVENT_CLASS:" in ln:
            return ln.split("EVENT_CLASS:", 1)[1].strip()
    return None

def fallback_flags(lines, wave_re):
    """
    Heuristic fallback if EVENT_CLASS is missing.
    Returns: has_W, has_P, has_H
    """
    has_wave = any(wave_re.search(ln) for ln in lines)

    # Picks
    has_pick_header = any(ln.startswith(" STAT") or ln.startswith("STAT ") for ln in lines)
    stationish = re.compile(r"^[A-Z0-9]{2,5}\s+[A-Z0-9]{1,3}\s+[A-Z0-9]{0,2}")
    has_station_lines = any(stationish.match(ln) for ln in lines)
    has_picks = has_pick_header or has_station_lines

    # Hypocenter
    has_gap = any("GAP=" in ln for ln in lines)
    has_type1 = any(ln.rstrip().endswith("1") and ln[:4].strip().isdigit() for ln in lines)
    has_hypo = has_gap or has_type1

    return has_wave, has_picks, has_hypo

def day_key_from_sfilename(name: str) -> str | None:
    """
    Parse date from SEISAN S-file name:
      DD-HHMM-SSL.SYYYYMM
    """
    m = re.search(r"^(?P<dd>\d{2})-.*\.S(?P<yyyy>\d{4})(?P<mm>\d{2})$", name)
    if not m:
        return None
    try:
        return f"{int(m.group('yyyy')):04d}-{int(m.group('mm')):02d}-{int(m.group('dd')):02d}"
    except Exception:
        return None

def run(
    *,
    rea_dir,
    out_dir,
    wavefile_regex='\\b[MS]\\.[A-Za-z0-9]{1,12}_[A-Za-z0-9]{1,6}\\b'
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        rea_dir=rea_dir,
        out_dir=out_dir,
        wavefile_regex=wavefile_regex,
    )

    rea_dir = Path(args.rea_dir)
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    wave_re = re.compile(args.wavefile_regex)

    combo = Counter()
    daily = defaultdict(lambda: {
        "total": 0,
        "wave": 0,
        "picks": 0,
        "hypo": 0
    })

    daily_by_class = defaultdict(lambda: Counter())

    sfiles = sorted(iter_sfiles(rea_dir))
    if not sfiles:
        raise SystemExit(f"No S-files found under {rea_dir}")

    for sf in sfiles:
        try:
            lines = sf.read_text(errors="replace").splitlines()
        except Exception:
            continue

        ev_class = get_event_class(lines)

        if ev_class:
            has_W = "W" in ev_class
            has_P = "P" in ev_class
            has_H = "H" in ev_class
            label = ev_class
        else:
            has_W, has_P, has_H = fallback_flags(lines, wave_re)
            label = (
                ("W" if has_W else "") +
                ("+P" if has_P else "") +
                ("+H" if has_H else "")
            ).lstrip("+") or "none"

        combo[label] += 1

        day = day_key_from_sfilename(sf.name)
        if not day:
            continue

        daily[day]["total"] += 1
        if has_W:
            daily[day]["wave"] += 1
        if has_P:
            daily[day]["picks"] += 1
        if has_H:
            daily[day]["hypo"] += 1

        daily_by_class[day][label] += 1

    # ------------------------------------------------------------------
    # OUTPUT: SUMMARY
    # ------------------------------------------------------------------

    print("\nSTEP 53 — EVENT CLASS SUMMARY")
    print("------------------------------")
    for k, v in combo.most_common():
        print(f"{k:10s}: {v}")

    # ------------------------------------------------------------------
    # OUTPUT: DAILY CSV
    # ------------------------------------------------------------------

    df_daily = pd.DataFrame.from_dict(daily, orient="index").sort_index()
    df_daily.index = pd.to_datetime(df_daily.index)
    df_daily.index.name = "date"

    df_class = pd.DataFrame.from_dict(daily_by_class, orient="index").fillna(0).astype(int)
    df_class.index = pd.to_datetime(df_class.index)
    df_class.index.name = "date"

    df = df_daily.join(df_class, how="left")

    csv_path = out_dir / "53_daily_counts.csv"
    df.to_csv(csv_path)

    # ------------------------------------------------------------------
    # OUTPUT: PLOT
    # ------------------------------------------------------------------

    fig_path = out_dir / "53_daily_counts.png"

    plt.figure(figsize=(10, 5))
    plt.plot(df.index, df["total"], label="total events")
    plt.plot(df.index, df["wave"], label="events with waveforms")
    plt.plot(df.index, df["picks"], label="events with picks")
    plt.plot(df.index, df["hypo"], label="events with hypocenters")
    plt.legend()
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.title("SEISAN REA Daily Event Counts (Step 53)")
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200)
    plt.close()

    print("\nOutputs")
    print("-------")
    print(f"Daily CSV : {csv_path}")
    print(f"Plot PNG  : {fig_path}")
    print("\nSTEP 53 COMPLETE")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## Run and inspect

Run this cell only after the configuration and all definitions. It writes the configured outputs; inspect the summary and exceptions before proceeding.

In [ ]:
result = run(**SETTINGS)

In [ ]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)

In [ ]:
from IPython.display import Image

# Show saved diagnostics inline; all figures remain in the configured folder.
plot_roots = [Path(SETTINGS[k]) for k in ('outdir', 'out_dir', 'out')
              if k in SETTINGS and SETTINGS[k] is not None]
plots = sorted({p for root in plot_roots
                for p in ([root] if root.suffix.lower() == '.png'
                          else root.rglob('*.png')) if p.is_file()})
print(f'{len(plots)} saved PNG figures; showing up to four.')
for plot in plots[:4]:
    print(plot)
    display(Image(filename=str(plot)))